In [ ]:

import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.regularizers import l1, l2

# Step 1: Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Step 2: Preprocess data
x_train = x_train[:5000].reshape(5000, 3072) / 255.0
y_train = y_train[:5000].flatten()

x_test = x_test[:1000].reshape(1000, 3072) / 255.0
y_test = y_test[:1000].flatten()

results = []

# Step 3: Define experiments
experiments = [
    ("Baseline", "None", "SGD"),
    ("Baseline", "None", "Adam"),
    ("Xavier", "None", "SGD"),
    ("Xavier", "None", "Adam"),
    ("Kaiming", "None", "SGD"),
    ("Kaiming", "None", "Adam"),
    ("Dropout", "Dropout (0.5)", "SGD"),
    ("Dropout", "Dropout (0.5)", "Adam"),
    ("L1", "L1 (0.001)", "SGD"),
    ("L1", "L1 (0.001)", "Adam"),
    ("L2", "L2 (0.001)", "SGD"),
    ("L2", "L2 (0.001)", "Adam")
]

# Step 4: Train and evaluate models
for name, reg, opt in experiments:

    # Weight initialization
    if name == "Kaiming":
        init = "he_normal"
    else:
        init = "glorot_uniform"

    # Regularization
    if name == "L1":
        regu = l1(0.001)
    elif name == "L2":
        regu = l2(0.001)
    else:
        regu = None

    # Build model
    model = Sequential([
        Input(shape=(3072,)),

        Dense(128, activation="relu",
              kernel_initializer=init,
              kernel_regularizer=regu),

        Dropout(0.5) if name == "Dropout" else tf.keras.layers.Activation("linear"),

        Dense(64, activation="relu",
              kernel_initializer=init,
              kernel_regularizer=regu),

        Dense(32, activation="relu",
              kernel_initializer=init,
              kernel_regularizer=regu),

        Dense(10, activation="softmax")
    ])

    # Compile model
    model.compile(
        optimizer=opt.lower(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Train model
    model.fit(
        x_train, y_train,
        epochs=3,
        batch_size=128,
        verbose=0
    )

    # Evaluate model
    loss, acc = model.evaluate(x_test, y_test, verbose=0)

    results.append([
        name, reg, opt,
        round(acc * 100, 2),
        round(loss, 4)
    ])

# Step 5: Display results
df = pd.DataFrame(
    results,
    columns=["Method", "Regularization", "Optimizer",
             "Accuracy (%)", "Loss"]
)

print(df)